# 2026-10-26 Transformed magnitudes and a real HR diagram

## Overview

+ Read two photometry files: `photometry.ecsv` (instrumental magnitudes) and `photometry-transformed.ecsv` (magnitudes transformed to the standard B and V system)
+ Make a color-magnitude diagram from each and compare them
+ Get distances from Gaia and turn the transformed V magnitude into an absolute magnitude
+ Compare the result to a *real* HR diagram

## The story so far

An **instrumental magnitude** is just $-2.5 \log_{10}(\text{counts}/\text{exposure time})$. It depends on the telescope, the camera, the filter *and* the atmosphere on the night you observed.

Each of those affects blue light and visual light differently:

+ the atmosphere scatters blue light more than red light, and the amount depends on airmass
+ our B filter and camera are not identical to the B filter and detector that *define* the standard B system (same for V)

So an instrumental $B - V$ is **not** the standard $B - V$. The **transformation** we ran last week uses stars in the field with known catalog magnitudes to fix two things:

1. the **horizontal** (color) scale, by fitting the difference between catalog and instrumental magnitudes as a function of color
2. the **vertical** (magnitude) scale, by setting the zero point so that our magnitudes agree with the catalog

Once the vertical scale is a standard V magnitude, a distance from Gaia turns it into an **absolute magnitude**, which is what a real HR diagram uses.

In [ ]:
from pathlib import Path
import numpy as np

from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table

from astroquery.gaia import Gaia

from stellarphot import PhotometryData
from ipyautoui.custom import FileChooser

%matplotlib inline
import matplotlib.pyplot as plt

## Read in the two photometry files

#### Choose your *instrumental* photometry file (should be called `photometry.ecsv`)

In [ ]:
p = Path("/calibrated/color-images-2026")
start = str(p) if p.exists() else "."
chooser_inst = FileChooser(filter_pattern=["*.ecsv"], path=start)
chooser_inst

#### Choose your *transformed* photometry file (should be called `photometry-transformed.ecsv`)

In [ ]:
chooser_transformed = FileChooser(filter_pattern=["*.ecsv"], path=start)
chooser_transformed

In [ ]:
inst_file = chooser_inst.value
transformed_file = chooser_transformed.value

In [ ]:
inst_table = PhotometryData.read(inst_file)
transformed_table = PhotometryData.read(transformed_file)

### What did the transformation add?

The transformed file has the same stars as the instrumental one, plus some new columns. The important ones are:

+ `mag_inst_cal` -- the transformed magnitude, on the standard system
+ `mag_cat` and `color_cat` -- the catalog magnitude and catalog $B - V$ for stars that were in the catalog

In [ ]:
sorted(set(transformed_table.colnames) - set(inst_table.colnames))

In [ ]:
transformed_V = transformed_table[transformed_table['passband'] == 'V']
transformed_V['star_id', 'passband', 'mag_inst', 'mag_inst_cal', 'mag_cat', 'color_cat'][:10]

## Split each table into B and V

This is the same thing we did last week, but wrapped in a function so we can do it to both tables. It keeps only stars that show up in *both* B and V, and sorts by `star_id` so that row $i$ of B and row $i$ of V are the same star.

In [ ]:
def split_BV(table):
    """
    Return B and V tables with the same stars in the same order.
    """
    B = table[table['passband'] == 'B']
    V = table[table['passband'] == 'V']

    common = set(B['star_id']) & set(V['star_id'])
    B = B[np.isin(B['star_id'], list(common))]
    V = V[np.isin(V['star_id'], list(common))]

    B.sort('star_id')
    V.sort('star_id')
    return B, V

In [ ]:
B_inst, V_inst = split_BV(inst_table)
B_transformed, V_transformed = split_BV(transformed_table)

len(B_inst), len(B_transformed)

## Color-magnitude diagrams: instrumental vs transformed

For the instrumental table the magnitude column is `mag_inst`. For the transformed table it is `mag_inst_cal`.

In [ ]:
color_inst = B_inst['mag_inst'] - V_inst['mag_inst']
mag_inst = V_inst['mag_inst']

color_transformed = B_transformed['mag_inst_cal'] - V_transformed['mag_inst_cal']
mag_transformed = V_transformed['mag_inst_cal']

In [ ]:
def plot_cmd(ax, bv, mag, xlabel="B - V", ylabel="V", title="", **kwargs):
    """
    Plot a color-magnitude diagram on the axes ax, with bright at the top.
    """
    kwargs.setdefault('marker', '.')
    kwargs.setdefault('linestyle', 'none')
    ax.plot(bv, mag, **kwargs)
    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(*ax.get_ylim()[::-1])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid()

In [ ]:
fig, (inst, transformed) = plt.subplots(1, 2, figsize=(12, 5))

plot_cmd(inst, color_inst, mag_inst,
         xlabel="Color (B - V instrumental)",
         ylabel="Apparent magnitude (V instrumental)",
         title="Instrumental")

plot_cmd(transformed, color_transformed, mag_transformed,
         xlabel="Color (B - V transformed)",
         ylabel="Apparent magnitude (V transformed)",
         title="Transformed")

fig.tight_layout()

### Questions

+ The two plots have the same horizontal limits. Where is the main clump of stars in each one? Did it move?
+ The vertical axes have very different numbers on them. Why? Did the *shape* of the diagram change vertically?

### Why does the color shift?

Plot the transformed color against the instrumental color for the same stars. If the atmosphere and telescope affected B and V identically, every star would fall on the dashed line.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

ax.plot(color_inst, color_transformed, '.', label='Your stars')
ax.plot([-0.5, 2.5], [-0.5, 2.5], '--', color='gray', label='Transformed = instrumental')
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 2.5)
ax.set_xlabel("B - V instrumental")
ax.set_ylabel("B - V transformed")
ax.set_aspect('equal')
ax.legend()
ax.grid()

The points do **not** fall on the dashed line. There is an offset (the zero points for B and V are different) and the slope is not exactly 1 (the *color term*: red stars and blue stars are affected by different amounts).

#### Sanity check: does the transformed color match the catalog?

Some of your stars are in the catalog that was used to find the transformation. For those we can compare the transformed color directly to the catalog color. This is the check we *could not* do with instrumental magnitudes.

In [ ]:
in_catalog = ~np.isnan(B_transformed['color_cat'])

fig, (cmp, resid) = plt.subplots(1, 2, figsize=(12, 5))

cmp.plot(B_transformed['color_cat'][in_catalog], color_transformed[in_catalog], '.')
cmp.plot([-0.5, 2.5], [-0.5, 2.5], '--', color='gray')
cmp.set_xlim(-0.5, 2.5)
cmp.set_ylim(-0.5, 2.5)
cmp.set_xlabel("Catalog B - V")
cmp.set_ylabel("Transformed B - V")
cmp.set_aspect('equal')
cmp.grid()

resid.plot(B_transformed['color_cat'][in_catalog], B_transformed['color_cat'][in_catalog] - color_transformed[in_catalog], '.')
resid.axhline(0, linestyle='--', color='gray')
resid.set_xlim(-0.5, 2.5)
resid.set_ylim(-0.5, 0.5)
resid.set_xlabel("Catalog B - V")
resid.set_ylabel("Catalog minus transformed B - V")
resid.grid()

fig.tight_layout()

## Correcting for distance

This is the same Gaia lookup as last week. We use the positions from the transformed V table, but the stars (and positions) are the same in both tables, so the match works for both.

In [ ]:
our_coord_V = SkyCoord(V_transformed['ra'], V_transformed['dec'])
center = SkyCoord(ra=our_coord_V.ra.mean(), dec=our_coord_V.dec.mean())
search_radius = our_coord_V.separation(center).max() + 1 * u.arcmin
center, search_radius.to(u.arcmin)

In [ ]:
Gaia.ROW_LIMIT = -1
j = Gaia.cone_search_async(center, radius=search_radius)
r = j.get_results()
gaia_cs = SkyCoord(ra=r['ra'], dec=r['dec'], unit='degree')
len(r)

In [ ]:
idxg, d2dg, _ = our_coord_V.match_to_catalog_sky(gaia_cs)
parallax = np.asarray(r['parallax'].filled(np.nan))[idxg]
great = (d2dg < 1 * u.arcsec) & (parallax > 0)
distance = 1000 / parallax  # parsecs, parallax is in milliarcsec
great.sum()

Absolute magnitude is $M = m - 5 (\log_{10} d - 1)$ with $d$ in parsecs. We do this to **both** the instrumental and the transformed V magnitudes so we can see what each step buys us.

In [ ]:
dist_mod = np.full(len(V_transformed), np.nan)
dist_mod[great] = 5 * (np.log10(distance[great]) - 1)

abs_mag_inst = mag_inst - dist_mod
abs_mag_transformed = mag_transformed - dist_mod

## Compare to a real HR diagram

The file `data/Hipparcos.csv` has about 900 nearby stars with measured $B - V$ and absolute V magnitude $M_V$ from the Hipparcos satellite. That is a real HR diagram in exactly the units we now have.

In [ ]:
hip = Table.read('data/Hipparcos.csv')
hip[:5]

In [ ]:
good = great & ~np.isnan(abs_mag_transformed) & ~np.isnan(abs_mag_inst)

fig, (inst, transformed, real) = plt.subplots(1, 3, figsize=(16, 5))

plot_cmd(inst, color_inst[good], abs_mag_inst[good],
         xlabel="Color (B - V instrumental)",
         ylabel="Absolute magnitude (V instrumental)",
         title="Instrumental color and mag, distance corrected")

plot_cmd(transformed, color_transformed[good], abs_mag_transformed[good],
         xlabel="Color (B - V transformed)",
         ylabel="Absolute magnitude (V transformed)",
         title="Transformed color and mag, distance corrected")
transformed.set_ylim(17, -5)

plot_cmd(real, hip['B-V'], hip['M_V'],
         xlabel="Color (B - V)",
         ylabel="Absolute magnitude (V)",
         title="Hipparcos stars (real HR diagram)",
         markersize=3, color='gray')
real.set_ylim(17, -5)

fig.tight_layout()

Look carefully at the numbers on the vertical axis of the left panel. Correcting for distance does not help if the magnitudes themselves have an arbitrary zero point: the *shape* is there, but the numbers mean nothing, and the colors are shifted to the red. The middle and right panels have the **same** axes, so those two can be compared directly.

### Your stars on top of the real HR diagram

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))

ax.plot(hip['B-V'], hip['M_V'], '.', color='lightgray', markersize=4, label='Hipparcos')
ax.plot(color_transformed[good], abs_mag_transformed[good], '.', label='Your stars (transformed)')
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(17, -5)
ax.set_xlabel("Color (B - V)")
ax.set_ylabel("Absolute magnitude (V)")
ax.set_title("Your stars on a real HR diagram")
ax.legend()
ax.grid()

### Questions

+ Where do most of your stars land on the Hipparcos diagram? Does that make sense for the kind of object you observed?
+ Compare the left and middle panels of the three-panel figure. The *only* difference is the horizontal axis. Which one lines up better with the real HR diagram?
+ What would have happened to the vertical axis if we had skipped the transformation and only corrected for distance?
+ Are there stars that land in places that make no sense (e.g. well below the main sequence)? What might be going on with those?

### For reference: the HR diagram from the Atlas of the Universe

In [ ]:
atlas_hr = plt.imread('media/atlas-universe-hr.png')

fig, (yours, atlas) = plt.subplots(1, 2, figsize=(14, 6))

plot_cmd(yours, color_transformed[good], abs_mag_transformed[good],
         xlabel="Color (B - V transformed)",
         ylabel="Absolute magnitude (V transformed)",
         title="Your stars")
yours.set_ylim(17, -5)

atlas.imshow(atlas_hr)
atlas.set_axis_off()
atlas.set_title("Atlas of the Universe")